In [1]:
!pip install requests
!pip install easyocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.2/307.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.3/908.3 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.3/281.3 kB 16.9 MB/s eta 0:00:00


In [2]:
import os
import requests
from io import BytesIO
from PIL import Image, UnidentifiedImageError
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from keras.applications import ResNet50
from keras.applications.resnet import preprocess_input
from keras.preprocessing import image

In [3]:
def download_images(df, output_dir="images"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for index, row in df.iterrows():
        image_url = row['image_link']
        try:
            response = requests.get(image_url)
            if 'image' not in response.headers.get('content-type', ''):
                print(f"URL at index {index} is not an image. Skipping.")
                continue

            img = Image.open(BytesIO(response.content))

            img.save(f"{output_dir}/{index}.jpg")
        except UnidentifiedImageError:
            print(f"Error: Unable to identify image at index {index} with URL {image_url}")
        except Exception as e:
            print(f"Failed to download or process image at index {index}. Error: {e}")

In [4]:
df = pd.read_csv("dataset/train.csv")

# Download the first 200 images for the training set
download_images(df)

In [5]:
import torch

# Check if GPU is available and its name
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available.")


No GPU available.


In [ ]:
import os
import easyocr

# Initialize the EasyOCR Reader for the desired languages (e.g., 'en' for English)
reader = easyocr.Reader(['en'])

# Function to extract text from a single image
def extract_text_from_image(image_path):
    print("\nextracting text from image")
    result = reader.readtext(image_path, detail=0)  # detail=0 returns only the text, not bounding boxes
    print("\nextracted text from image")
    return ' '.join(result)

# Function to iterate through all images in the 'images' folder and extract text
def extract_text_from_images_folder(folder_path):
    extracted_texts = {}

    # Iterate through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith(".jpg") or filename.endswith(".png"):  # Only process image files
            image_path = os.path.join(folder_path, filename)
            try:
                text = extract_text_from_image(image_path)
                extracted_texts[filename] = text  # Store the filename and its extracted text
            except Exception as e:
                print(f"Error extracting text from {filename}: {e}")

    return extracted_texts

# Usage example
folder_path = 'images'  # Replace with your images folder path
extracted_texts = extract_text_from_images_folder(folder_path)
print("hello we are here")
# Print extracted texts from all images
for image, text in extracted_texts.items():
    print(f"Text from {image}: {text}")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.10/dist-packages/easyocr/detection.py:78: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(copyStateDict(torch.load(trained_model, ma


extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extracting text from image

extracted text from image

extractin

In [ ]:
import re
import csv

def extract_max_weight(text):
    # Define regex patterns for common weight units
    patterns = [
        r'\b(\d+\.?\d*)\s*(kg|kilos|kilograms|lbs|pounds|g|grams)\b',  # Matches kg, kilos, kilograms, lbs, pounds, g, grams
    ]

    max_weight = 0
    max_weight_str = ''

    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            value, unit = match
            value = float(value)
            # Keep track of the maximum weight without converting to kg
            if value > max_weight:
                max_weight = value
                max_weight_str = f"{value} {unit}"

    return max_weight_str







In [ ]:
# Extract text from dictionary values
texts = [text for text in extracted_texts.values()]

# Define the output CSV file path
csv_file_path = 'weights.csv'

# Write results to CSV
with open(csv_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Index', 'Max Weight'])
    for image, text in extracted_texts.items():
      max_weight = extract_max_weight(text)
      writer.writerow([image, max_weight])

print(f"Results have been written to {csv_file_path}")

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('weights.csv')

# Ensure the index column is treated as integers for sorting
df['Index'] = df['Index'].str.extract('(\d+)').astype(float)

# Sort by the index
df = df.sort_values(by='Index')

# Write the sorted DataFrame back to CSV
df.to_csv('sorted_weights.csv', index=False)


In [ ]:
import pandas as pd
import re

# Function to clean and normalize dimensions
def clean_and_normalize(value):
    # Ensure the value is a string
    if not isinstance(value, str):
        return 'Invalid'

    # Define regex patterns for valid units
    patterns = {
        'g': r'\b\d+(\.\d+)?\s?g\b',
        'kg': r'\b\d+(\.\d+)?\s?kg\b',
        'cm': r'\b\d+(\.\d+)?\s?cm\b',
        'meter': r'\b\d+(\.\d+)?\s?meter\b',
        'ounce': r'\b\d+(\.\d+)?\s?ounce\b'
    }

    # Normalize the input
    value = value.strip().lower()

    # Check if the value matches any valid patterns
    for unit, pattern in patterns.items():
        if re.fullmatch(pattern, value):
            return value

    # Fallback or correction for some common invalid cases
    fallback_patterns = {
        r'\b(\d+(\.\d+)?)\s*(gms|grams)\b': r'\1 g',
        r'\b(\d+(\.\d+)?)\s*(kg|kilogram|kilograms)\b': r'\1 kg',
        r'\b(\d+(\.\d+)?)\s*(ounce)\b': r'\1 ounce',
        r'\b(\d+(\.\d+)?)\s*(cm|centimeter|centimeters)\b': r'\1 cm',
        r'\b(\d+(\.\d+)?)\s*(meter|meters)\b': r'\1 meter'
    }

    # Try to apply fallback patterns
    for invalid_pattern, correct_format in fallback_patterns.items():
        if re.fullmatch(invalid_pattern, value):
            return re.sub(invalid_pattern, correct_format, value)

    # If still invalid, return a placeholder or default value
    return 'Invalid'

# Read the CSV file
df = pd.read_csv('weights.csv')

# Convert all entries to string and apply cleaning and normalization
df['Max Weight'] = df['Max Weight'].astype(str).apply(clean_and_normalize)

# Write the cleaned DataFrame back to CSV
df.to_csv('cleaned_weights.csv', index=False)



In [ ]:
import pandas as pd

def check_column_names(df, expected_columns):
    """Check if the DataFrame contains the expected columns."""
    return set(df.columns) == set(expected_columns)

def check_data_types(df, expected_types):
    """Check if the DataFrame columns have the expected data types."""
    return all(df[col].dtype == expected_types[col] for col in expected_types)

def check_valid_values(df, column, valid_values):
    """Check if all values in a column are within the valid set of values."""
    return df[column].apply(lambda x: x in valid_values).all()

def sanity_check(file_path):
    """Perform sanity checks on the output file."""
    # Define expected columns and their types
    expected_columns = ['Index', 'Max Weight']
    expected_types = {'Index': 'float64', 'Max Weight': 'object'}
    valid_units = {'g', 'kg', 'cm', 'meter', 'ounce'}

    # Load the DataFrame
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        return f"Error reading CSV file: {e}"

    # Check column names
    if not check_column_names(df, expected_columns):
        return "Column names do not match expected columns."

    # Check data types
    if not check_data_types(df, expected_types):
        return "Data types do not match expected types."

    # Check valid values in 'Max Weight'
    if not check_valid_values(df, 'Max Weight', valid_units):
        return "Some values in 'Max Weight' column are invalid."

    return "Sanity check passed."

# Example usage
if __name__ == "__main__":
    result = sanity_check('cleaned_weights.csv')
    print(result)


In [ ]:
!python sanity.py --test_filename sample_test.csv --output_filename sample_test_out.csv

In [ ]:
import pandas as pd

# Define the input and output file paths
input_csv = 'sorted_weights.csv'  # Replace with your input file
output_csv = 'updated_weights.csv'  # Replace with your desired output file

# Load the CSV file into a DataFrame
df = pd.read_csv(input_csv)

# Rename the column from 'Max Weight' to 'predictions'
df.rename(columns={'Index': 'index'}, inplace=True)
df.rename(columns={'Max Weight': 'prediction'}, inplace=True)
# def replace_g_with_grams(value):
#     if isinstance(value, str):
#         return value.replace(' g', ' gram')
#     return value
def replace_units(value):
    if isinstance(value, str):
        # Replace 'g' with 'grams'
        value = value.replace(' g', ' gram')
        # Replace 'kg' with 'kilograms'
        value = value.replace(' kg', ' kilogram')
        value = value.replace(' Kg', ' kilogram')
        value = value.replace(' KG', ' kilogram')
        value = value.replace(' G', ' gram')
        value = value.replace(' gramRAMS', ' gram')
        value = value.replace(' gramrams', ' gram')
        value = value.replace(' lbs', ' pound')
        value = value.replace(' LBS', ' pound')
    return value

# Apply the replacement function to the 'predictions' column
df['prediction'] = df['prediction'].apply(replace_units)

# Save the updated DataFrame back to a CSV file
df.to_csv(output_csv, index=False)

print(f"Column 'Max Weight' has been renamed to 'prediction' and saved to {output_csv}.")

In [ ]:

!python sanity.py --test_filename sample_test.csv --output_filename updated_weights.csv


In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('sorted_weights.csv')

# Print the columns to check their names
print(df.columns)


Volume

In [ ]:
def extract_volume_units(text):
    # Define a regex pattern for common volume units
    volume_units_pattern = r'\b(\d*\.?\d+)\s*(liters?|l|milliliters?|ml|gallons?|gal|cubic meters?|m³|cubic centimeters?|cm³|pints?|pt|quarts?|qt|centilitres?|cl|decilitres?|dl|fluid ounces?|fl oz|cups?)\b'
    # Find all matches in the text
    matches = re.findall(volume_units_pattern, text, re.IGNORECASE)

    # Format the results
    extracted_units = [(value, unit) for value, unit in matches]

    return extracted_units

In [ ]:
# Extract volume units from the sample text
# Usage example
extracted_texts = extract_text_from_images_folder(folder_path)

# Print extracted texts from all images
for image, text in extracted_texts.items():
    # Call extract_volume_units for each text
    extracted_volume_units = extract_volume_units(text)

    # Print the results for the current image
    print(f"Extracted units from {image}:")
    for value, unit in extracted_volume_units:
        print(f"  Value: {value}, Unit: {unit}")

In [ ]:
def extract_max_volume(text):
    # Define regex patterns for common volume units
    patterns = [
    r'\b(\d+\.?\d*)\s*(liters?|l|milliliters?|ml|gallons?|gal|cubic meters?|m³|cubic centimeters?|cm³|pints?|pt|quarts?|qt|centilitres?|cl|decilitres?|dl|fluid ounces?|fl oz|cups?)\b',  # Matches various volume units
]

    max_volume = 0
    max_volume_str = ''

    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            value, unit = match
            value = float(value)
            # Keep track of the maximum volume without converting to liters
            if value > max_volume:
                max_volume = value
                max_volume_str = f"{value} {unit}"

    return max_volume_str

In [ ]:
for image, text in extracted_texts.items():
    # Call extract_volume_units for each text
    extracted_volume_units = extract_volume_units(text)

    # Call extract_max_volume for each text
    max_volume = extract_max_volume(text)

    # Print the results for the current image
    print(f"Extracted units from {image}:")
    for value, unit in extracted_volume_units:
        print(f"  Value: {value}, Unit: {unit}")

    print(f"Maximum volume extracted: {max_volume}")

In [ ]:
import csv

# Define the output CSV file path
csv_file_path = 'weights_and_volumes.csv'

# Write results to CSV
with open(csv_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Image', 'Max Weight', 'Max Volume'])  # Updated header

    for image, text in extracted_texts.items():
        max_weight = extract_max_weight(text)
        max_volume = extract_max_volume(text)

        # Write the image name, max weight, and max volume in one row
        writer.writerow([image, max_weight, max_volume])

print(f"Results have been written to {csv_file_path}")

In [ ]:
import csv

# Define the output CSV file path
csv_file_path = 'weights_and_volumes.csv'

# Write results to CSV
with open(csv_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Image', 'Max Weight/Volume'])  # Updated header

    for image, text in extracted_texts.items():
        max_weight = extract_max_weight(text)
        max_volume = extract_max_volume(text)

        # Determine whether to write max weight or max volume
        if 'kg' in max_weight or 'lbs' in max_weight:
            result = max_weight
        else:
            result = max_volume

        # Write the image name and max weight/volume in one row
        writer.writerow([image, result])

print(f"Results have been written to {csv_file_path}")

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('weights_and_volumes.csv')

# Ensure the index column is treated as integers for sorting
df['Image'] = df['Image'].str.extract('(\d+)').astype(float)

# Sort by the index
df = df.sort_values(by='Image')

# Write the sorted DataFrame back to CSV
df.to_csv('sorted_weights_and_volumes.csv', index=False)


In [ ]:
import pandas as pd
import re

# Function to clean and normalize volume dimensions
def clean_and_normalize_volume(value):
    # Ensure the value is a string
    if not isinstance(value, str):
        return 'Invalid'

    # Define regex patterns for valid volume units
    patterns = {
    'l': r'\b\d+(\.\d+)?\s?l\b',
    'liters': r'\b\d+(\.\d+)?\s?liters?\b',
    'ml': r'\b\d+(\.\d+)?\s?ml\b',
    'milliliters': r'\b\d+(\.\d+)?\s?milliliters?\b',
    'gal': r'\b\d+(\.\d+)?\s?gal\b',
    'gallons': r'\b\d+(\.\d+)?\s?gallons?\b',
    'm³': r'\b\d+(\.\d+)?\s?m³\b',
    'cubic meters': r'\b\d+(\.\d+)?\s?cubic meters?\b',
    'cm³': r'\b\d+(\.\d+)?\s?cm³\b',
    'cubic centimeters': r'\b\d+(\.\d+)?\s?cubic centimeters?\b',
    'pt': r'\b\d+(\.\d+)?\s?pt\b',
    'pints': r'\b\d+(\.\d+)?\s?pints?\b',
    'qt': r'\b\d+(\.\d+)?\s?qt\b',
    'quarts': r'\b\d+(\.\d+)?\s?quarts?\b',
    'centilitre': r'\b\d+(\.\d+)?\s?centilitres?\b',
    'cl': r'\b\d+(\.\d+)?\s?cl\b',
    'cubic foot': r'\b\d+(\.\d+)?\s?cubic feet?\b',
    'cubic inch': r'\b\d+(\.\d+)?\s?cubic inches?\b',
    'cup': r'\b\d+(\.\d+)?\s?cups?\b',
    'decilitre': r'\b\d+(\.\d+)?\s?decilitres?\b',
    'dl': r'\b\d+(\.\d+)?\s?dl\b',
    'fluid ounce': r'\b\d+(\.\d+)?\s?fluid ounces?\b',
    'fl oz': r'\b\d+(\.\d+)?\s?fl oz\b'}

    # Normalize the input
    value = value.strip().lower()

    # Check if the value matches any valid patterns
    for unit, pattern in patterns.items():
        if re.fullmatch(pattern, value):
            return value

    # Fallback or correction for some common invalid cases
    fallback_patterns = {
    r'\b(\d+(\.\d+)?)\s*(liters|liter)\b': r'\1 l',
    r'\b(\d+(\.\d+)?)\s*(ml|milliliters|milliliter)\b': r'\1 ml',
    r'\b(\d+(\.\d+)?)\s*(gallons|gallon)\b': r'\1 gal',
    r'\b(\d+(\.\d+)?)\s*(cubic meters|cubic meter|m³)\b': r'\1 m³',
    r'\b(\d+(\.\d+)?)\s*(cubic centimeters|cubic centimeter|cm³)\b': r'\1 cm³',
    r'\b(\d+(\.\d+)?)\s*(pints|pint)\b': r'\1 pt',
    r'\b(\d+(\.\d+)?)\s*(quarts|quart)\b': r'\1 qt',
    r'\b(\d+(\.\d+)?)\s*(centilitres|centilitre|cl)\b': r'\1 cl',
    r'\b(\d+(\.\d+)?)\s*(cubic feet|cubic foot)\b': r'\1 cubic foot',
    r'\b(\d+(\.\d+)?)\s*(cubic inches|cubic inch)\b': r'\1 cubic inch',
    r'\b(\d+(\.\d+)?)\s*(cups|cup)\b': r'\1 cup',
    r'\b(\d+(\.\d+)?)\s*(decilitres|decilitre|dl)\b': r'\1 dl',
    r'\b(\d+(\.\d+)?)\s*(fluid ounces|fluid ounce|fl oz)\b': r'\1 fl oz'}

    # Try to apply fallback patterns
    for invalid_pattern, correct_format in fallback_patterns.items():
        if re.fullmatch(invalid_pattern, value):
            return re.sub(invalid_pattern, correct_format, value)

    # If still invalid, return a placeholder or default value
    return 'Invalid'

# Read the CSV file
df = pd.read_csv('sorted_weights_and_volumes.csv')

# Convert all entries to string and apply cleaning and normalization
df['Max Weight/Volume'] = df['Max Weight/Volume'].astype(str).apply(clean_and_normalize_volume)

# Write the cleaned DataFrame back to CSV
df.to_csv('cleaned_volumes.csv', index=False)

print("Volume units have been cleaned and written to 'cleaned_volumes.csv'.")

In [ ]:
import pandas as pd
import re

# Function to clean and normalize dimensions (both volume and weight)
def clean_and_normalize(value):
    # Ensure the value is a string
    if not isinstance(value, str):
        return 'Invalid'

    # Define regex patterns for valid volume units
    volume_patterns = {
        'l': r'\b\d+(\.\d+)?\s?l\b',
        'liters': r'\b\d+(\.\d+)?\s?liters?\b',
        'ml': r'\b\d+(\.\d+)?\s?ml\b',
        'milliliters': r'\b\d+(\.\d+)?\s?milliliters?\b',
        'gal': r'\b\d+(\.\d+)?\s?gal\b',
        'gallons': r'\b\d+(\.\d+)?\s?gallons?\b',
        'm³': r'\b\d+(\.\d+)?\s?m³\b',
        'cubic meters': r'\b\d+(\.\d+)?\s?cubic meters?\b',
        'cm³': r'\b\d+(\.\d+)?\s?cm³\b',
        'cubic centimeters': r'\b\d+(\.\d+)?\s?cubic centimeters?\b',
        'pt': r'\b\d+(\.\d+)?\s?pt\b',
        'pints': r'\b\d+(\.\d+)?\s?pints?\b',
        'qt': r'\b\d+(\.\d+)?\s?qt\b',
        'quarts': r'\b\d+(\.\d+)?\s?quarts?\b',
        'centilitre': r'\b\d+(\.\d+)?\s?centilitres?\b',
        'cl': r'\b\d+(\.\d+)?\s?cl\b',
        'cubic foot': r'\b\d+(\.\d+)?\s?cubic feet?\b',
        'cubic inch': r'\b\d+(\.\d+)?\s?cubic inches?\b',
        'cup': r'\b\d+(\.\d+)?\s?cups?\b',
        'decilitre': r'\b\d+(\.\d+)?\s?decilitres?\b',
        'dl': r'\b\d+(\.\d+)?\s?dl\b',
        'fluid ounce': r'\b\d+(\.\d+)?\s?fluid ounces?\b',
        'fl oz': r'\b\d+(\.\d+)?\s?fl oz\b'
    }

    # Define regex patterns for valid weight units
    weight_patterns = {
        'g': r'\b\d+(\.\d+)?\s?g\b',
        'kg': r'\b\d+(\.\d+)?\s?kg\b',
        'microgram': r'\b\d+(\.\d+)?\s?microgram(s)?\b',
        'milligram': r'\b\d+(\.\d+)?\s?milligram(s)?\b',
        'ounce': r'\b\d+(\.\d+)?\s?ounce(s)?\b',
        'pound': r'\b\d+(\.\d+)?\s?pound(s)?\b',
        'ton': r'\b\d+(\.\d+)?\s?ton(s)?\b'
    }

    # Normalize the input
    value = value.strip().lower()

    # Check if the value matches any valid volume patterns
    for unit, pattern in {**volume_patterns, **weight_patterns}.items():
        if re.fullmatch(pattern, value):
            return value

    # Fallback or correction for some common invalid cases
    fallback_patterns = {
        r'\b(\d+(\.\d+)?)\s*(liters|liter)\b': r'\1 l',
        r'\b(\d+(\.\d+)?)\s*(ml|milliliters|milliliter)\b': r'\1 ml',
        r'\b(\d+(\.\d+)?)\s*(gallons|gallon)\b': r'\1 gal',
        r'\b(\d+(\.\d+)?)\s*(cubic meters|cubic meter|m³)\b': r'\1 m³',
        r'\b(\d+(\.\d+)?)\s*(cubic centimeters|cubic centimeter|cm³)\b': r'\1 cm³',
        r'\b(\d+(\.\d+)?)\s*(pints|pint)\b': r'\1 pt',
        r'\b(\d+(\.\d+)?)\s*(quarts|quart)\b': r'\1 qt',
        r'\b(\d+(\.\d+)?)\s*(centilitres|centilitre|cl)\b': r'\1 cl',
        r'\b(\d+(\.\d+)?)\s*(cubic feet|cubic foot)\b': r'\1 cubic foot',
        r'\b(\d+(\.\d+)?)\s*(cubic inches|cubic inch)\b': r'\1 cubic inch',
        r'\b(\d+(\.\d+)?)\s*(cups|cup)\b': r'\1 cup',
        r'\b(\d+(\.\d+)?)\s*(decilitres|decilitre|dl)\b': r'\1 dl',
        r'\b(\d+(\.\d+)?)\s*(fluid ounces|fluid ounce|fl oz)\b': r'\1 fl oz',
        r'\b(\d+(\.\d+)?)\s*(grams|gram)\b': r'\1 g',
        r'\b(\d+(\.\d+)?)\s*(kilograms|kilogram)\b': r'\1 kg',
        r'\b(\d+(\.\d+)?)\s*(micrograms|microgram)\b': r'\1 microgram',
        r'\b(\d+(\.\d+)?)\s*(milligrams|milligram)\b': r'\1 milligram',
        r'\b(\d+(\.\d+)?)\s*(ounces|ounce)\b': r'\1 ounce',
        r'\b(\d+(\.\d+)?)\s*(pounds|pound)\b': r'\1 pound',
        r'\b(\d+(\.\d+)?)\s*(tons|ton)\b': r'\1 ton'
    }

    # Try to apply fallback patterns
    for invalid_pattern, correct_format in fallback_patterns.items():
        if re.fullmatch(invalid_pattern, value):
            return re.sub(invalid_pattern, correct_format, value)

    # If still invalid, return a placeholder or default value
    return 'Invalid'

# Read the CSV file
df = pd.read_csv('sorted_weights_and_volumes.csv')

# Convert all entries to string and apply cleaning and normalization
df['Max Weight/Volume'] = df['Max Weight/Volume'].astype(str).apply(clean_and_normalize)

# Write the cleaned DataFrame back to CSV
df.to_csv('cleaned_volumes.csv', index=False)

print("Volume and weight units have been cleaned and written to 'cleaned_volumes.csv'.")

In [ ]:
import pandas as pd

# Sample DataFrame
data = pd.read_csv('cleaned_volumes.csv')
output_csv = 'cleaned_volumes.csv'
# Create a DataFrame from the sample data
df = pd.DataFrame(data)

# Change column name from 'Max Weight' to 'Max Volume'
df.rename(columns={'Image':'index','Max Weight/Volume': 'prediction'}, inplace=True)

# Display updated DataFrame
print("\nUpdated DataFrame:")

df.to_csv(output_csv, index=False)

In [ ]:
import pandas as pd

def check_column_names(df, expected_columns):
    """Check if the DataFrame contains the expected columns."""
    return set(df.columns) == set(expected_columns)

def check_data_types(df, expected_types):
    """Check if the DataFrame columns have the expected data types."""
    return all(df[col].dtype == expected_types[col] for col in expected_types)

def check_valid_values(df, column, valid_values):
    """Check if all values in a column are within the valid set of values."""
    return df[column].apply(lambda x: x in valid_values).all()

def sanity_check(file_path):
    """Perform sanity checks on the output file."""
    # Define expected columns and their types
    expected_columns = ['index', 'prediction']  # Updated to 'Max Volume'
    expected_types = {'index': 'float64', 'prediction': 'object'}  # Updated to 'Max Volume'
    valid_units = {'l', 'liters', 'ml', 'milliliters', 'gal', 'gallons',
               'm³', 'cubic meters', 'cm³', 'cubic centimeters',
               'pt', 'pints', 'qt', 'quarts',
               'centilitre', 'centilitres', 'cl',
               'cubic foot', 'cubic inch',
               'cup', 'cups',
               'decilitre', 'decilitres', 'dl',
               'fluid ounce', 'fluid ounces', 'fl oz'} # Updated valid units for volume

    count=0
    # Load the DataFrame
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        return f"Error reading CSV file: {e}"

    # Check column names
    if not check_column_names(df, expected_columns):
        return "Column names do not match expected columns."

    # Check data types
    if not check_data_types(df, expected_types):
        return "Data types do not match expected types."

    # Check valid values in 'Max Volume'
    if not check_valid_values(df, 'prediction', valid_units):
        count= count+1
        return f"Some values in 'Max Volume' column are invalid.with count{count}"

    return "Sanity check passed."

# Example usage
if __name__ == "__main__":
    result = sanity_check('cleaned_volumes.csv')  # Updated file name
    print(result)

In [ ]:
import pandas as pd
import re

# Load the DataFrame from CSV
data = pd.read_csv('cleaned_volumes.csv')
df = pd.DataFrame(data)

# Mapping of short forms to full forms
unit_conversion = {
    r'\bl\b': 'litre',  # Use regex to match whole words
    r'\bml\b': 'millilitre',
    r'\bgal\b': 'gallon',
    r'\bgallons\b': 'gallon',
    r'\bm³\b': 'cubic meter',
    r'\bcm³\b': 'cubic centimetre',
    r'\bpt\b': 'pint',
    r'\bpints\b': 'pint',
    r'\bqt\b': 'quart',
    r'\bquarts\b': 'quart',
    r'\bcl\b': 'centilitre',
    r'\bcentilitres\b': 'centilitre',
    r'\bcup\b': 'cup',
    r'\bcups\b': 'cup',
    r'\bdl\b': 'decilitre',
    r'\bdecilitres\b': 'decilitre',
    r'\bfl oz\b': 'fluid ounce',
    r'\bfluid ounces\b': 'fluid ounce',
    r'\bin³\b': 'cubic inch',
    r'\bcubic inches\b': 'cubic inch',
    r'\bcubic foot\b': 'cubic foot',
    r'\bcubic feet\b': 'cubic foot'
}

# Function to convert short forms to full forms
def convert_units(df, column):
    """Convert short forms of units in a specified column to full forms."""
    for short, full in unit_conversion.items():
        # Replace using regex to ensure we match whole words
        df[column] = df[column].replace(short, full, regex=True)
    return df

# Convert short forms in the 'Max Volume' column
df = convert_units(df, 'prediction')

# Rename columns and save to CSV
df.rename(columns={'Index': 'index', 'prediction': 'prediction'}, inplace=True)

# Define output CSV file path
output_csv = 'cleaned_volumes.csv'
df.to_csv(output_csv, index=False)

# Display the updated DataFrame
print("Updated DataFrame:")


In [ ]:
import pandas as pd
import re

# Load the DataFrame from CSV
data = pd.read_csv('cleaned_volumes.csv')
df = pd.DataFrame(data)

# Mapping of short forms to full forms for both volume and weight
unit_conversion = {
    r'\bl\b': 'litre',  # Use regex to match whole words
    r'\bml\b': 'millilitre',
    r'\bgal\b': 'gallon',
    r'\bgallons\b': 'gallon',
    r'\bm³\b': 'cubic meter',
    r'\bcm³\b': 'cubic centimetre',
    r'\bpt\b': 'pint',
    r'\bpints\b': 'pint',
    r'\bqt\b': 'quart',
    r'\bquarts\b': 'quart',
    r'\bcl\b': 'centilitre',
    r'\bcentilitres\b': 'centilitre',
    r'\bcup\b': 'cup',
    r'\bcups\b': 'cup',
    r'\bdl\b': 'decilitre',
    r'\bdecilitres\b': 'decilitre',
    r'\bfl oz\b': 'fluid ounce',
    r'\bfluid ounces\b': 'fluid ounce',
    r'\bin³\b': 'cubic inch',
    r'\bcubic inches\b': 'cubic inch',
    r'\bcubic foot\b': 'cubic foot',
    r'\bcubic feet\b': 'cubic foot',
    r'\bg\b': 'gram',
    r'\bkg\b': 'kilogram',
    r'\bμg\b': 'microgram',
    r'\bmg\b': 'milligram',
    r'\boz\b': 'ounce',
    r'\bpound\b': 'pound',
    r'\bton\b': 'ton',
    r'\btons\b': 'ton'
}

# Function to convert short forms to full forms
def convert_units(df, column):
    """Convert short forms of units in a specified column to full forms."""
    for short, full in unit_conversion.items():
        # Replace using regex to ensure we match whole words
        df[column] = df[column].replace(short, full, regex=True)
    return df

# Convert short forms in the 'Max Volume' column
df = convert_units(df, 'prediction')

# Rename columns and save to CSV
df.rename(columns={'Index': 'index', 'prediction': 'prediction'}, inplace=True)

# Define output CSV file path
output_csv = 'cleaned_volumes.csv'
df.to_csv(output_csv, index=False)

# Display the updated DataFrame
print("Updated DataFrame:")
print(df)

In [ ]:
# prompt: in the table wherever the value is Invalid change it to empty

import pandas as pd
# Load the DataFrame from CSV
data = pd.read_csv('cleaned_volumes.csv')
df = pd.DataFrame(data)

# Replace 'Invalid' values with empty strings in the 'prediction' column
df['prediction'] = df['prediction'].replace('Invalid', '', regex=True)

# Define output CSV file path
output_csv = 'cleaned_volumes.csv'
df.to_csv(output_csv, index=False)

# Display the updated DataFrame
print("Updated DataFrame:")



In [ ]:
!python sanity.py --test_filename sample_test.csv --output_filename cleaned_volumes.csv